In [ ]:
import pandas as pd
import tensorflow as tf

df = pd.read_csv('/home/jsoychak/cs/ecg/data/mitbih_train.csv')
df = df.sample(1024)

def prepare_data(data: pd.DataFrame):
    x = data.to_numpy()
    x, y = x[:, :-1], x[:, -1]
    x = tf.expand_dims(x, axis=-1)

    y = tf.cast(y, dtype=tf.int32)
    y = tf.one_hot(y, 5)
    
    return x, y

x_train, y_train = prepare_data(df)

In [ ]:
df_y_train = pd.DataFrame(y_train.numpy(), columns=['N', 'S', 'E', 'F', 'Q'])
counts = df_y_train.sum()
weights = (1.0/counts)*(len(df_y_train)/len(counts))

In [ ]:
from benchmark_model import BenchmarkModel

model = BenchmarkModel(32, 5)
model.summary()

In [ ]:
epochs = 8
batch_size = 16

In [ ]:
import keras

model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[keras.metrics.CategoricalAccuracy()]
)
model.fit(
    x=x_train,
    y=y_train,
    batch_size=batch_size,
    epochs=epochs,
    class_weight={i: w for i, w in enumerate(weights)}
)

In [ ]:
df_test = pd.read_csv('/home/jsoychak/cs/ecg/data/mitbih_test.csv')

x_test, y_test = prepare_data(df_test)
model.evaluate(x_test, y_test)

In [ ]:
model.save('benchmark.keras')